# 👂 Train Virgo's own Khmer hearing — Kaggle (free GPU)

Fine-tunes **Whisper large-v3-turbo** (Virgo's speech to text) on Khmer speech, so Virgo understands spoken Khmer better. Data: **OpenSLR 42** (Google's Khmer recordings, CC BY-SA 4.0), **FLEURS Khmer** when it can be loaded, and your own recordings if you add them.

**First time only**
1. Kaggle → **Settings → Phone verification**.
2. **Add-ons → Secrets**: `HF_TOKEN` (a Hugging Face token with **write** access), ticked for this notebook.

**Run**
1. **Session options**: **Accelerator → GPU T4 x2**, **Internet → On**.
2. **Save Version → Save & Run All (Commit)**. It runs in the background (about 3–5 hours), so you can close the tab.
3. It prints the Khmer **error rate (CER) before and after** on clips it never trained on. If it improved, it's saved privately as **Virgo-1.0-Angkor-Hearing**, and the server notebooks use it automatically.

**Your own recordings (optional):** make a Kaggle dataset of `.wav` files and a `metadata.csv` (`file_name,sentence`), add it to this notebook, and set `EXTRA` below. Everyday speech from many different people helps the most.

In [ ]:
# ⚙️ Settings
MODEL_NAME = "Virgo-1.0-Angkor-Hearing"
STEPS = 2000      # training steps (about 3–5 hours on a T4)
EXTRA = ""        # optional: folder of your own recordings (.wav + metadata.csv)

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: Session options → Accelerator → GPU T4 x2"
WORK = "/kaggle/working"
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"])
    assert r.returncode == 0, "❌ Can't download: Session options → Internet → On (needs phone verification)"
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt", "librosa", "soundfile"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft
from huggingface_hub import login, HfApi
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
assert token, "❌ Add the HF_TOKEN secret (Add-ons → Secrets) and tick it for this notebook"
login(token=token)
REPO = f"{HfApi().whoami()['name']}/{MODEL_NAME}"
print("✅ Ready. Virgo's hearing will be saved to", REPO)

def run(cmd):
    """Runs a step and shows its output here; on failure, shows the last lines (send them to Claude)."""
    import subprocess, collections
    tail = collections.deque(maxlen=60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        tail.append(line)
    if proc.wait() != 0:
        print("\n" + "=" * 60 + "\n❌ It stopped with an error. Send these last lines to Claude:\n" + "=" * 60)
        print("".join(tail))
        raise SystemExit(f"❌ {next((a for a in cmd if a.endswith('.py')), cmd[0])} failed (exit code {proc.returncode})")


In [ ]:
# 2️⃣ Train (prints the Khmer error rate before and after)
import os, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cmd = [sys.executable, "-u", "speech/finetune_stt.py", "--steps", str(STEPS), "--push", REPO, "--work", "/kaggle/working/stt"]
if EXTRA:
    cmd += ["--extra", EXTRA]
run(cmd)
print(open("speech/out/virgo-1.0-stt/score.txt").read())